# RAG System with OpenRouter - Windows Compatible

This notebook demonstrates a Retrieval-Augmented Generation (RAG) system using:
- **Windows-compatible setup** (no GPU requirements)
- **OpenRouter's free Gemma 4 31B model** (no local downloads)
- **FAISS vector store** for efficient retrieval
- **Sentence transformers** for embeddings

## Benefits:
- ✅ Works on any Windows PC
- ✅ No large model downloads (saves 4.4GB)
- ✅ Free AI model via OpenRouter
- ✅ Same RAG functionality as original

## Installing and Importing Necessary Libraries

In [1]:
%pip install -r requirements_windows.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Note**:
- This installation is optimized for Windows and CPU usage only
- No GPU dependencies or CUDA libraries required
- All packages are compatible with Windows environments

In [ ]:
from langchain.chains import RetrievalQA
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain_community.chat_models import ChatOpenAI
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

c:\Users\jaych\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configure OpenRouter API

In [ ]:
load_dotenv()

llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="google/gemma-4-31b-it",
    temperature=0.01,
    max_tokens=2048
)

print(f"OpenRouter LLM initialized: google/gemma-4-31b-it")
print(f"API Key configured: {'Yes' if os.getenv('OPENROUTER_API_KEY') else 'No - Set OPENROUTER_API_KEY in .env'}")

OpenRouter LLM initialized: google/gemma-4-31b-it
API Key configured: Yes


## Load the data

In [4]:
import os

# Use absolute path for file loading with UTF-8 encoding
file_path = r"c:\Users\jaych\Documents\UT_Agents_Course_2026\UT-Agents-Course\Module 1\AAPL-MDA.txt"

loader = TextLoader(file_path, encoding='utf-8')
data = loader.load()
print(f"Loaded {len(data)} documents from: {file_path}")

Loaded 1 documents from: c:\Users\jaych\Documents\UT_Agents_Course_2026\UT-Agents-Course\Module 1\AAPL-MDA.txt


## Split the Extracted Data into Text Chunks

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
text_chunks = text_splitter.split_documents(data)
print(f"Created {len(text_chunks)} text chunks")

Created 715 text chunks


In [6]:
print(f"Total chunks: {len(text_chunks)}")

Total chunks: 715


In [7]:
print("Sample chunk content:")
print(text_chunks[3].page_content[:500] + "...")

Sample chunk content:
and Apple Watch. Services Services net sales increased during 2020 compared to 2019 due primarily to higher net sales from the App Store, advertising and cloud services. Apple Inc. | 2020 Form 10-K | 21 Segment Operating Performance The Company manages its business primarily on a geographic basis. The Company’s reportable segments consist of the Americas, Europe, Greater China, Japan and Rest of Asia Pacific. Americas includes both North and South America. Europe includes European countries, as ...


## Load the embedding model

In [8]:
print("Loading embedding model...")
embeddings = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")
print("Embedding model loaded successfully")

Loading embedding model...
Embedding model loaded successfully


## Create Vector Store

In [9]:
print("Creating vector store...")
vector_store = FAISS.from_documents(text_chunks, embeddings)
print("Vector store created successfully")
print(f"Indexed {len(text_chunks)} document chunks")

Creating vector store...
Vector store created successfully
Indexed 715 document chunks


## Build the RAG Chain

In [10]:
print("Building RAG chain...")
agent_colab = RetrievalQA.from_chain_type(
    llm=llm,
    verbose=True,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 2}),
    chain_type_kwargs={"verbose": True}
)
print("RAG chain created successfully")

Building RAG chain...
RAG chain created successfully


## Test the RAG System

In [11]:
query = "How often does the company review inventory, and what is considered in this inventory calculation?"
print(f"Test Query: {query}")

Test Query: How often does the company review inventory, and what is considered in this inventory calculation?


In [12]:
print("Processing query...")
output_colab = agent_colab.run(query)
print("\n" + "="*50)
print("ANSWER:")
print("="*50)
print(output_colab)

Processing query...


> Entering new RetrievalQA chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

have to adjust its allowance for doubtful accounts, which would affect earnings in the period the adjustments were made. Inventory Valuation and Inventory Purchase Commitments The Company must order components for its products and build inventory in advance of product shipments. The Company records a write-down for inventories of components and products, including third-party products held for resale, which have become obsolete or are in excess of anticipated demand or net realizable value. The Company performs a detailed review of inventory each fiscal quarter that considers multiple factors including demand forecasts, product life cycle status, product developme

BadRequestError: Error code: 400 - {'error': {'message': 'Input required: specify "prompt"', 'code': 400}, 'user_id': 'user_3CXvqVpe7QmLtUgqhYnL19t3cnx'}

In [ ]:
print("\n" + "="*60)
print("FINAL ANSWER:")
print("="*60)
print(output_colab)
print("="*60)